# Embeddings + ML — Prediction from Sentence Embeddings

## What it does
Uses **pre-computed sentence embeddings** (e.g. from BERT, Sentence-Transformers, or any
dense text representation) as predictors for macro/financial outcomes. The pipeline:

1. **Load embeddings** — Parquet/CSV file where each row is one document with a date and a
   fixed-length embedding vector
2. **Aggregate to monthly** — average all document embeddings within each calendar month
3. **Merge with outcomes** — align the monthly mean embedding to the outcome time series
4. **Model horse race** — fit LASSO, Ridge, and KernelRidge (RBF) with inner CV tuning;
   compare by validation OOS R²
5. **Report and save** — model comparison table, coefficient plots (for linear models),
   actual vs fitted plot

## Why embeddings?
Pre-trained language model embeddings capture **semantic similarity** that bag-of-words
(TF-IDF) misses. Two articles about 'rate hike' and 'monetary tightening' are far apart
in TF-IDF space but close in embedding space.

Limitations: embeddings are dense (250–768 dims), so ridge-type regularization is needed;
they are harder to interpret than sparse LASSO word selections.

## Embedding file format
- Parquet or CSV with one row per document/article
- Must contain: a **date column** and **N numeric columns** (the embedding dimensions)
- If article-level: multiple rows per date are allowed — they will be averaged to monthly
- Embedding dimensions can be named anything (e.g. `dim_0, dim_1, ...` or `e0, e1, ...`)

## Outcome file format
Same as other NLP notebooks: CSV with a date column and one or more numeric outcome columns.

## Configuration

In [ ]:
CONFIG = {
    # --- Embeddings file ---
    'EMBEDDINGS_FILE':    '../../data/embeddings.parquet',  # parquet or csv
    'EMB_DATE_COL':       'date',
    # Column name pattern that identifies embedding dimensions.
    # Set to None to auto-detect (all non-date numeric columns).
    'EMB_COL_PREFIX':     None,    # e.g. 'dim_' to select dim_0, dim_1, ...
    # --- Outcome file ---
    'OUTCOME_FILE':       '../../data/macro.csv',
    'OUTCOME_DATE_COL':   'date',
    'TARGET_COLS':        ['mret', 'vol'],
    # --- Train/val/test split (fraction-based, preserving time order) ---
    'TRAIN_FRAC':         0.6,
    'VAL_FRAC':           0.2,
    # --- Models to include ---
    'RUN_LASSO':          True,
    'RUN_RIDGE':          True,
    'RUN_KERNEL_RIDGE':   True,
    # --- Hyperparameter grids ---
    'LASSO_ALPHAS':       [1e-4, 1e-3, 0.01, 0.1, 1.0],
    'RIDGE_ALPHAS':       [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0],
    'KR_ALPHAS':          [0.01, 0.1, 1.0, 10.0, 100.0],
    'KR_GAMMAS':          [0.001, 0.01, 0.1, 1.0],
    # --- Output ---
    'SAVE_RESULTS':       True,
    'OUTPUT_DIR':         'results',
}

print('Configuration loaded.')
for k, v in CONFIG.items():
    print(f'  {k}: {v}')

## Step 1 — Load Embeddings & Aggregate to Monthly

In [ ]:
import sys, warnings, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import Lasso, Ridge
from sklearn.kernel_ridge import KernelRidge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path('../../').resolve()))
from utils import load_csv, load_parquet, compute_oos_r2

# Load embeddings
if CONFIG['EMBEDDINGS_FILE'].endswith(('.pq', '.parquet')):
    emb_df = load_parquet(CONFIG['EMBEDDINGS_FILE'])
else:
    emb_df = load_csv(CONFIG['EMBEDDINGS_FILE'], parse_dates=[CONFIG['EMB_DATE_COL']])

emb_df[CONFIG['EMB_DATE_COL']] = pd.to_datetime(emb_df[CONFIG['EMB_DATE_COL']]).dt.to_period('M').dt.to_timestamp()

# Identify embedding columns
if CONFIG['EMB_COL_PREFIX'] is not None:
    emb_cols = [c for c in emb_df.columns if c.startswith(CONFIG['EMB_COL_PREFIX'])]
else:
    emb_cols = [c for c in emb_df.columns
                if c != CONFIG['EMB_DATE_COL'] and pd.api.types.is_numeric_dtype(emb_df[c])]

print(f'Raw embeddings  : {emb_df.shape[0]:,} rows')
print(f'Embedding dims  : {len(emb_cols)}')
print(f'Date range      : {emb_df[CONFIG["EMB_DATE_COL"]].min().date()} — {emb_df[CONFIG["EMB_DATE_COL"]].max().date()}')

In [ ]:
# Aggregate to monthly mean
emb_monthly = (
    emb_df.groupby(CONFIG['EMB_DATE_COL'])[emb_cols]
    .mean()
    .reset_index()
    .rename(columns={CONFIG['EMB_DATE_COL']: 'date'})
)

print(f'Monthly periods : {len(emb_monthly)}')
print(f'Embedding dims  : {len(emb_cols)}')
emb_monthly.head()

## Step 2 — Load Outcomes & Merge

In [ ]:
outcome_df = load_csv(CONFIG['OUTCOME_FILE'], parse_dates=[CONFIG['OUTCOME_DATE_COL']])
outcome_df[CONFIG['OUTCOME_DATE_COL']] = pd.to_datetime(outcome_df[CONFIG['OUTCOME_DATE_COL']]).dt.to_period('M').dt.to_timestamp()

targets = [CONFIG['TARGET_COLS']] if isinstance(CONFIG['TARGET_COLS'], str) else CONFIG['TARGET_COLS']

df = emb_monthly.merge(
    outcome_df[[CONFIG['OUTCOME_DATE_COL']] + targets].rename(columns={CONFIG['OUTCOME_DATE_COL']: 'date'}),
    on='date', how='inner',
).sort_values('date').reset_index(drop=True)

print(f'Merged sample   : {len(df)} months')
print(f'Features        : {len(emb_cols)} embedding dims')
print(f'Targets         : {targets}')
df.head()

## Step 3 — Train / Val / Test Split & Standardize

In [ ]:
n       = len(df)
n_train = int(n * CONFIG['TRAIN_FRAC'])
n_val   = int(n * CONFIG['VAL_FRAC'])

X_all  = df[emb_cols].values
dates  = df['date'].values

X_train = X_all[:n_train]
X_val   = X_all[n_train:n_train + n_val]
X_test  = X_all[n_train + n_val:]

scaler   = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

print(f'Train : {n_train} months  ({df["date"].iloc[0].date()} — {df["date"].iloc[n_train-1].date()})')
print(f'Val   : {n_val} months  ({df["date"].iloc[n_train].date()} — {df["date"].iloc[n_train+n_val-1].date()})')
print(f'Test  : {n - n_train - n_val} months  ({df["date"].iloc[n_train+n_val].date()} — {df["date"].iloc[-1].date()})')

## Step 4 — Model Horse Race per Target

For each target: grid search selects hyperparameters by **validation OOS R²**.
Test set is never used during tuning.

In [ ]:
all_results = {}

for target in targets:
    print(f'\n{"="*65}')
    print(f'TARGET: {target}')
    print('='*65)

    y_all_t  = df[target].values.astype(float)
    y_train  = y_all_t[:n_train]
    y_val    = y_all_t[n_train:n_train + n_val]
    y_test   = y_all_t[n_train + n_val:]
    bench    = float(y_train.mean())

    target_results = {}

    # --- LASSO ---
    if CONFIG['RUN_LASSO']:
        best_a, best_r2 = None, -np.inf
        for a in CONFIG['LASSO_ALPHAS']:
            m = Lasso(alpha=a, max_iter=10000).fit(X_train_s, y_train)
            r2 = compute_oos_r2(y_val, m.predict(X_val_s), bench)
            if r2 > best_r2:
                best_r2, best_a = r2, a
        m_lasso = Lasso(alpha=best_a, max_iter=10000).fit(X_train_s, y_train)
        n_nz    = int(np.sum(m_lasso.coef_ != 0))
        test_r2 = compute_oos_r2(y_test, m_lasso.predict(X_test_s), bench)
        target_results['LASSO'] = {'model': m_lasso, 'best_param': f'alpha={best_a}',
                                    'val_oos_r2': best_r2, 'test_oos_r2': test_r2,
                                    'extra': f'n_nonzero={n_nz}'}
        print(f'  LASSO     alpha={best_a}  Val OOS R²={best_r2*100:+.4f}%  '
              f'Test OOS R²={test_r2*100:+.4f}%  nonzero={n_nz}')

    # --- Ridge ---
    if CONFIG['RUN_RIDGE']:
        best_a, best_r2 = None, -np.inf
        for a in CONFIG['RIDGE_ALPHAS']:
            m = Ridge(alpha=a).fit(X_train_s, y_train)
            r2 = compute_oos_r2(y_val, m.predict(X_val_s), bench)
            if r2 > best_r2:
                best_r2, best_a = r2, a
        m_ridge = Ridge(alpha=best_a).fit(X_train_s, y_train)
        test_r2 = compute_oos_r2(y_test, m_ridge.predict(X_test_s), bench)
        target_results['Ridge'] = {'model': m_ridge, 'best_param': f'alpha={best_a}',
                                    'val_oos_r2': best_r2, 'test_oos_r2': test_r2, 'extra': ''}
        print(f'  Ridge     alpha={best_a}  Val OOS R²={best_r2*100:+.4f}%  '
              f'Test OOS R²={test_r2*100:+.4f}%')

    # --- KernelRidge ---
    if CONFIG['RUN_KERNEL_RIDGE']:
        best_params, best_r2 = None, -np.inf
        for a in CONFIG['KR_ALPHAS']:
            for g in CONFIG['KR_GAMMAS']:
                m = KernelRidge(alpha=a, kernel='rbf', gamma=g).fit(X_train_s, y_train)
                r2 = compute_oos_r2(y_val, m.predict(X_val_s), bench)
                if r2 > best_r2:
                    best_r2, best_params = r2, (a, g)
        m_kr = KernelRidge(alpha=best_params[0], kernel='rbf',
                           gamma=best_params[1]).fit(X_train_s, y_train)
        test_r2 = compute_oos_r2(y_test, m_kr.predict(X_test_s), bench)
        target_results['KernelRidge'] = {
            'model': m_kr,
            'best_param': f'alpha={best_params[0]}, gamma={best_params[1]}',
            'val_oos_r2': best_r2, 'test_oos_r2': test_r2, 'extra': ''}
        print(f'  KernelRidge alpha={best_params[0]}, gamma={best_params[1]}  '
              f'Val OOS R²={best_r2*100:+.4f}%  Test OOS R²={test_r2*100:+.4f}%')

    all_results[target] = target_results

## Step 5 — Comparison Table & Plots

In [ ]:
for target in targets:
    res = all_results[target]
    y_all_t = df[target].values.astype(float)
    y_test  = y_all_t[n_train + n_val:]
    test_dates = dates[n_train + n_val:]

    # Summary table
    rows = []
    for mname, mr in res.items():
        rows.append({'Model': mname, 'Best param': mr['best_param'],
                     'Val OOS R²': f"{mr['val_oos_r2']*100:+.4f}%",
                     'Test OOS R²': f"{mr['test_oos_r2']*100:+.4f}%",
                     'Extra': mr['extra']})
    print(f'\n{target} — Summary')
    print(pd.DataFrame(rows).to_string(index=False))

    # Actual vs predicted (test, best model by val OOS R²)
    best_mname = max(res.keys(), key=lambda m: res[m]['val_oos_r2'])
    best_model = res[best_mname]['model']
    test_pred  = best_model.predict(X_test_s)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].plot(test_dates, y_test,   color='black',   linewidth=1.5, label='Actual')
    axes[0].plot(test_dates, test_pred, color='steelblue', linewidth=1.2, linestyle='--', label=best_mname)
    axes[0].set(title=f'{target} — Actual vs {best_mname} (test)', ylabel=target)
    axes[0].legend(fontsize=9)
    axes[0].grid(True, alpha=0.3)

    # OOS R² bar chart across models
    model_names_t = list(res.keys())
    r2_vals = [res[m]['test_oos_r2'] * 100 for m in model_names_t]
    bar_cols = ['steelblue' if v >= 0 else 'tomato' for v in r2_vals]
    axes[1].barh(model_names_t, r2_vals, color=bar_cols, alpha=0.8)
    axes[1].axvline(x=0, color='black', linewidth=0.8)
    axes[1].set(xlabel='Test OOS R² (%)', title=f'{target} — Model Comparison')
    axes[1].grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.show()

    # Ridge coefficient plot (top embedding dimensions)
    if 'Ridge' in res:
        m_ridge = res['Ridge']['model']
        top_n   = 20
        coef_df = pd.DataFrame({'dim': emb_cols, 'coef': m_ridge.coef_})
        coef_df = coef_df.reindex(coef_df['coef'].abs().nlargest(top_n).index)
        fig, ax = plt.subplots(figsize=(8, 5))
        colors = ['steelblue' if v > 0 else 'tomato' for v in coef_df['coef']]
        ax.barh(range(top_n), coef_df['coef'].values, color=colors, alpha=0.8)
        ax.set_yticks(range(top_n))
        ax.set_yticklabels(coef_df['dim'].values, fontsize=8)
        ax.invert_yaxis()
        ax.axvline(x=0, color='black', linewidth=0.5)
        ax.set(xlabel='Ridge Coefficient', title=f'Top {top_n} Embedding Dims → {target} (Ridge)')
        ax.grid(True, alpha=0.3, axis='x')
        plt.tight_layout()
        plt.show()

## Step 6 — Save Results

In [ ]:
if CONFIG['SAVE_RESULTS']:
    os.makedirs(CONFIG['OUTPUT_DIR'], exist_ok=True)

    rows = []
    for target in targets:
        for mname, mr in all_results[target].items():
            rows.append({
                'target':       target,
                'model':        mname,
                'best_param':   mr['best_param'],
                'val_oos_r2':   mr['val_oos_r2'],
                'test_oos_r2':  mr['test_oos_r2'],
                'extra':        mr['extra'],
                'emb_dims':     len(emb_cols),
                'n_months':     len(df),
                'train_frac':   CONFIG['TRAIN_FRAC'],
                'val_frac':     CONFIG['VAL_FRAC'],
                'notebook':     'embeddings_ml',
            })

    path = os.path.join(CONFIG['OUTPUT_DIR'], 'embeddings_ml_summary.csv')
    pd.DataFrame(rows).to_csv(path, index=False)
    print(f'Saved: {path}')
else:
    print('SAVE_RESULTS = False — skipping.')